In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
mus = [0, 0, 4, 4]
thetas = [1, 4, 1, 4]

In [3]:
def est_1(x):
    return np.var(x, ddof=1)

In [4]:
def est_2(x):
    median = np.median(x)
    return np.sum((x - median)**2)/(len(x)-1)

In [5]:
def est_3(x):
    return np.median((x - np.mean(x))**2)

In [6]:
n = 50
m = 10**4

estimators = [est_1, est_2, est_3]

In [7]:
def generate_sample(mu, theta, size):
    return mu + np.sqrt(theta) * np.random.randn(size)

In [8]:
def theta_stats(estimates, true_theta):
    estimates = np.array(estimates)
    mean_est = np.mean(estimates)
    var_est = np.var(estimates, ddof=1) # żeby dzieliło prze n-ddof
    bias = mean_est - true_theta
    mse = np.mean((estimates - true_theta)**2)
    return {"Mean": mean_est, "Variance": var_est, "Bias": bias, "MSE": mse}

In [10]:
import os
import pandas as pd

os.makedirs("plots", exist_ok=True)

results = {}

for mu, theta in zip(mus, thetas):
    estimates = {f"est_{i+1}": [] for i in range(len(estimators))}
    
    for _ in range(m):
        data = generate_sample(mu, theta, n)
        for i, est in enumerate(estimators):
            estimates[f"est_{i+1}"].append(est(data))
    
    est_stats = {name: theta_stats(values, theta) for name, values in estimates.items()}
    results[(mu, theta)] = est_stats

    plt.figure(figsize=(10, 6))
    plt.boxplot(
        [estimates[f"est_{i+1}"] for i in range(len(estimators))],
        tick_labels=[f"est_{i+1}" for i in range(len(estimators))],
        patch_artist=True
    )
    plt.title(f"Rozkład estymatorów wariancji (mu={mu}, theta={theta})")
    plt.ylabel("Szacowana wariancja")
    plt.grid(True)

    all_vals = np.hstack([v for v in estimates.values()])
    q1, q3 = np.percentile(all_vals, [25, 75])
    iqr = q3 - q1
    delta = 0.5
    plt.ylim(-delta + q1 - 2*iqr, q3 + 2*iqr + delta)

    png_filename = f"plots/boxplot_mu_{mu}_theta_{theta}.png"
    plt.savefig(png_filename)
    # plt.show()
    plt.close()

    stats_dict = {
        "Estimator": [],
        "Mean": [],
        "Variance": [],
        "Bias": [],
        "MSE": []
    }

    for i, (est, stats) in enumerate(est_stats.items()):
        stats_dict["Estimator"].append(est)
        for (key, value) in stats.items():
            stats_dict[key].append(value)

    df_stats = pd.DataFrame(stats_dict)

    # zaokrąglamy do 2 miejsc po przecinku
    df_stats = df_stats.round(2)

    # zapis do CSV przez pandas
    csv_filename = f"plots/stats_mu_{mu}_theta_{theta}.csv"
    df_stats.to_csv(csv_filename, index=False)

    print(f"Zapisano wykres: {png_filename} i statystyki: {csv_filename}")


Zapisano wykres: plots/boxplot_mu_0_theta_1.png i statystyki: plots/stats_mu_0_theta_1.csv
Zapisano wykres: plots/boxplot_mu_0_theta_4.png i statystyki: plots/stats_mu_0_theta_4.csv
Zapisano wykres: plots/boxplot_mu_4_theta_1.png i statystyki: plots/stats_mu_4_theta_1.csv
Zapisano wykres: plots/boxplot_mu_4_theta_4.png i statystyki: plots/stats_mu_4_theta_4.csv
